In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import * 
from pyspark.sql.types import *

spark=SparkSession.builder.appName("revision").getOrCreate()

In [2]:
data = [
(101, "tarun@123", "tarun@gmail.com", "9876543210", "new york", "NY", "Laptop", "Electronics", 2, 55000, "Card", "2025-01-10", "Delivered"),
(102, "ALI$$", "ali@yahoo.com", "9123456780", None, "TX", "Phone", "Electronics", 1, 15000, "UPI", "2025-01-12", "Pending"),
(103, "raFi_45", "rafi@gmail.com", "9988776655", "New Jersey", "NJ", "Shoes", "Fashion", 3, 2500, "Cash", "2025-01-13", "Delivered"),
(104, "uday#K", "uday@gmail.com", "9876501234", "NEW DELHI", "DL", "Watch", "Accessories", 5, 2000, "Card", "2025-01-14", "Cancelled"),
(105, "mike99", "mike@hotmail.com", "9123409876", "chicago", "IL", "Tablet", "Electronics", 1, 22000, "Card", "2025-01-15", "Delivered"),
(106, "sara@!", "sara@gmail.com", "9999988888", None, "CA", "Bag", "Fashion", 4, 1800, "UPI", "2025-01-16", "Delivered"),
(107, "johnDoe", "john@gmail.com", "9111122222", "New York", "NY", "Camera", "Electronics", 2, 30000, "Card", "2025-01-17", "Pending"),
(108, "anil_kumar", "anil@gmail.com", "9000011111", "newark", "NJ", "Headphones", "Electronics", 6, 1500, "Cash", "2025-01-18", "Delivered"),
(109, "Priya#", "priya@gmail.com", "9888877777", "New Delhi", "DL", "Shoes123", "Fashion", 2, 3000, "Card", "2025-01-19", "Delivered"),
(110, "KIRAN@%", "kiran@yahoo.com", "9777766666", "Houston", "TX", "TV", "Electronics", 1, 45000, "UPI", "2025-01-20", "Pending"),
(111, "rohit_1", "rohit@gmail.com", "9666655555", "new jersey", "NJ", "Watch", "Accessories", 3, 2500, "Cash", "2025-01-21", "Delivered"),
(112, "Meena$$", "meena@gmail.com", "9555544444", None, "CA", "Laptop", "Electronics", 1, 60000, "Card", "2025-01-22", "Delivered")
]

columns = ["order_id", "customer_name", "email", "phone", "city", "state",
           "product", "category", "quantity", "price",
           "payment_method", "order_date", "delivery_status"]

orders_df = spark.createDataFrame(data, columns)

orders_df.show(truncate=False)

+--------+-------------+----------------+----------+----------+-----+----------+-----------+--------+-----+--------------+----------+---------------+
|order_id|customer_name|email           |phone     |city      |state|product   |category   |quantity|price|payment_method|order_date|delivery_status|
+--------+-------------+----------------+----------+----------+-----+----------+-----------+--------+-----+--------------+----------+---------------+
|101     |tarun@123    |tarun@gmail.com |9876543210|new york  |NY   |Laptop    |Electronics|2       |55000|Card          |2025-01-10|Delivered      |
|102     |ALI$$        |ali@yahoo.com   |9123456780|NULL      |TX   |Phone     |Electronics|1       |15000|UPI           |2025-01-12|Pending        |
|103     |raFi_45      |rafi@gmail.com  |9988776655|New Jersey|NJ   |Shoes     |Fashion    |3       |2500 |Cash          |2025-01-13|Delivered      |
|104     |uday#K       |uday@gmail.com  |9876501234|NEW DELHI |DL   |Watch     |Accessories|5       

In [3]:
"""
✅ 1. Clean the data
Remove special characters from customer_name
Convert name to initcap
Extract username from email
Convert city to uppercase
Replace null city with "UNKNOWN"
"""

df_cleaned = (
    orders_df.withColumn(
        "customer_name", regexp_replace(col("customer_name"), "[^a-zA-Z]", "")
    )
    .withColumn("customer_name", initcap(col("customer_name")))
    .withColumn("username", split(col("email"), "@")[0])
    .withColumn("city", upper(col("city")))
    .withColumn("city", coalesce(col("city"), lit("UNKNOWN")))
)

In [4]:
df_cleaned.show(truncate=False)

+--------+-------------+----------------+----------+----------+-----+----------+-----------+--------+-----+--------------+----------+---------------+--------+
|order_id|customer_name|email           |phone     |city      |state|product   |category   |quantity|price|payment_method|order_date|delivery_status|username|
+--------+-------------+----------------+----------+----------+-----+----------+-----------+--------+-----+--------------+----------+---------------+--------+
|101     |Tarun        |tarun@gmail.com |9876543210|NEW YORK  |NY   |Laptop    |Electronics|2       |55000|Card          |2025-01-10|Delivered      |tarun   |
|102     |Ali          |ali@yahoo.com   |9123456780|UNKNOWN   |TX   |Phone     |Electronics|1       |15000|UPI           |2025-01-12|Pending        |ali     |
|103     |Rafi         |rafi@gmail.com  |9988776655|NEW JERSEY|NJ   |Shoes     |Fashion    |3       |2500 |Cash          |2025-01-13|Delivered      |rafi    |
|104     |Udayk        |uday@gmail.com  |98765

In [5]:
"""
✅ 2. Create new columns
productDetails → product + " by " + category
order_value → quantity * price
discount_flag:
If order_value > 10000 → "HIGH"
Between 5000 and 10000 → "MEDIUM"
Otherwise → "LOW"
"""

df_transformed=(
    df_cleaned.withColumn("productDetails",concat_ws(" by ",col("product"),col("category")))
    .withColumn("order_value",col("quantity")*col("price"))
    .withColumn("discount_flag",
    when(col("order_value")>1000,lit("HIGH"))
    .when(col("order_value").between(5000,10000),lit("MEDIUM"))
    .otherwise(lit("LOW"))
    )
)

df_transformed.show(truncate=False)

+--------+-------------+----------------+----------+----------+-----+----------+-----------+--------+-----+--------------+----------+---------------+--------+-----------------------+-----------+-------------+
|order_id|customer_name|email           |phone     |city      |state|product   |category   |quantity|price|payment_method|order_date|delivery_status|username|productDetails         |order_value|discount_flag|
+--------+-------------+----------------+----------+----------+-----+----------+-----------+--------+-----+--------------+----------+---------------+--------+-----------------------+-----------+-------------+
|101     |Tarun        |tarun@gmail.com |9876543210|NEW YORK  |NY   |Laptop    |Electronics|2       |55000|Card          |2025-01-10|Delivered      |tarun   |LaptopbyElectronics    |110000     |HIGH         |
|102     |Ali          |ali@yahoo.com   |9123456780|UNKNOWN   |TX   |Phone     |Electronics|1       |15000|UPI           |2025-01-12|Pending        |ali     |Phoneb

In [6]:
# Filtering
filtered_df = df_transformed.filter(
    # Name starts with lowercase (before cleaning exam may ask raw column)
    col("customer_name").rlike("^[A-Z]")
    &
    # City like NEW%
    col("city").like("NEW%")
    &
    # Gmail users
    (split(col("email"), "@")[1] == "gmail.com")
    &
    # Product only alphabets
    col("product").rlike("^[A-Za-z]+$")
)

In [7]:
# 4️⃣ GROUPING + AGGREGATION
# ---------------------------

grouped_df = filtered_df.groupBy("city", "discount_flag").agg(
    count("order_id").alias("total_orders"),
    sum("quantity").alias("total_quantity"),
    sum("order_value").alias("total_revenue"),
    first("product").alias("first_product"),
    last("product").alias("last_product"),
)

# ---------------------------
# 5️⃣ SORTING
# ---------------------------

sorted_df = grouped_df.sort(col("total_revenue").desc())
sorted_df.show(truncate=False)

+----------+-------------+------------+--------------+-------------+-------------+------------+
|city      |discount_flag|total_orders|total_quantity|total_revenue|first_product|last_product|
+----------+-------------+------------+--------------+-------------+-------------+------------+
|NEW YORK  |HIGH         |2           |4             |170000       |Laptop       |Camera      |
|NEW JERSEY|HIGH         |2           |6             |15000        |Shoes        |Watch       |
|NEW DELHI |HIGH         |1           |5             |10000        |Watch        |Watch       |
|NEWARK    |HIGH         |1           |6             |9000         |Headphones   |Headphones  |
+----------+-------------+------------+--------------+-------------+-------------+------------+



In [8]:
# Clean customer_name:

# Remove all non-alphabet characters

# Convert to proper case (InitCap)

# Create new column name_length (length of cleaned name)

qs1=(
    orders_df.withColumn("customer_name",regexp_replace(col("customer_name"),"[^a-zA-Z]",""))
    .withColumn("customer_name",initcap(col("customer_name")))
    .withColumn("name_length",length(col("customer_name")))
)

In [12]:
qsn2 = (
    qs1.withColumn("Username", split(col("email"), "@")[0])
    .withColumn("Domain", split(col("email"), "@")[1])
    .withColumn("Username", lower(col("Username")))
    .filter(col("Domain") == "gmail.com")
)

qsn3 = (
    qsn2.withColumn("city", coalesce(col("city"), lit("UNKNOWN")))
    .withColumn("city", upper(col("city")))
    .withColumn("city_prefix", substring(col("city"), 1, 3))  # FIXED
)

qsn4 = qsn3.filter(
    col("product").rlike("^[A-Za-z]+$")  # FIXED
    & (col("quantity") > 2)
    & col("price").between(2000, 50000)
    & col("city").like("NEW%")
)

qsn4.show()

+--------+-------------+---------------+----------+----------+-----+-------+-----------+--------+-----+--------------+----------+---------------+-----------+--------+---------+-----------+
|order_id|customer_name|          email|     phone|      city|state|product|   category|quantity|price|payment_method|order_date|delivery_status|name_length|Username|   Domain|city_prefix|
+--------+-------------+---------------+----------+----------+-----+-------+-----------+--------+-----+--------------+----------+---------------+-----------+--------+---------+-----------+
|     103|         Rafi| rafi@gmail.com|9988776655|NEW JERSEY|   NJ|  Shoes|    Fashion|       3| 2500|          Cash|2025-01-13|      Delivered|          4|    rafi|gmail.com|        NEW|
|     104|        Udayk| uday@gmail.com|9876501234| NEW DELHI|   DL|  Watch|Accessories|       5| 2000|          Card|2025-01-14|      Cancelled|          5|    uday|gmail.com|        NEW|
|     111|        Rohit|rohit@gmail.com|9666655555|NEW 

In [ ]:
# & has higher precendence than |
qsn5 = qsn4.filter(
    (
        col("customer_name").rlike("^[a-z]") |
        (col("Domain") != "gmail.com")
    ) &
    (col("payment_method") == "Card")
)

qsn5.show(truncate=False)

+--------+-------------+-----+-----+----+-----+-------+--------+--------+-----+--------------+----------+---------------+-----------+--------+------+-----------+
|order_id|customer_name|email|phone|city|state|product|category|quantity|price|payment_method|order_date|delivery_status|name_length|Username|Domain|city_prefix|
+--------+-------------+-----+-----+----+-----+-------+--------+--------+-----+--------------+----------+---------------+-----------+--------+------+-----------+
+--------+-------------+-----+-----+----+-----+-------+--------+--------+-----+--------------+----------+---------------+-----------+--------+------+-----------+



In [19]:
qsn6=qsn4.groupBy("city","category").agg(
    count("*").alias("total_orders"),
    sum(col("quantity")).alias("total_quantity"),
    sum(col("price")*col("quantity")).alias("Total_revenue"),
    avg(col("price")).alias("Average_Price"),
    first(col("product")).alias("firstProduct"),
    last(col("product")).alias("lastProduct")
).orderBy(col("Total_revenue").desc())
qsn6.show(truncate=False)

+----------+-----------+------------+--------------+-------------+-------------+------------+-----------+
|city      |category   |total_orders|total_quantity|Total_revenue|Average_Price|firstProduct|lastProduct|
+----------+-----------+------------+--------------+-------------+-------------+------------+-----------+
|NEW DELHI |Accessories|1           |5             |10000        |2000.0       |Watch       |Watch      |
|NEW JERSEY|Accessories|1           |3             |7500         |2500.0       |Watch       |Watch      |
|NEW JERSEY|Fashion    |1           |3             |7500         |2500.0       |Shoes       |Shoes      |
+----------+-----------+------------+--------------+-------------+-------------+------------+-----------+



In [ ]:
# first we need to count number of unique products >2
from pyspark.sql.functions import countDistinct
qsn7 = (
    qsn4.groupBy("city")
    .agg(
        sum(col("price") * col("quantity")).alias("Total_revenue"),
        countDistinct("product").alias("unique_products"),
    )
    .filter((col("Total_revenue") > 100000) & (col("unique_products") > 2))
)

qsn7.show()

sqlquery="""
select city,sum(price*quantity) as total_revenue,count(distinct product) as unique_products
from table group by city having total_revenue>100000 and unique_products>2
"""

+----+-------------+---------------+
|city|Total_revenue|unique_products|
+----+-------------+---------------+
+----+-------------+---------------+



In [22]:
qsn8 = (
    qsn4.withColumn("order_value", col("price") * col("quantity"))
    .withColumn(
        "Order_Segment",
        when(col("order_value") > 50000, "PREMIUM")
        .when(col("order_value").between(20000, 50000), "GOLD")
        .when(col("order_value").between(5000, 20000), "SILVER")
        .otherwise("BASIC"),
    )
    .groupBy("Order_Segment")
    .agg(sum("order_value").alias("TotalRevenue"))
)

qsn8.show()

+-------------+------------+
|Order_Segment|TotalRevenue|
+-------------+------------+
|       SILVER|       25000|
+-------------+------------+



In [ ]:
from pyspark.sql.functions import *

qsn9 = (
    qsn4.withColumn("order_value", col("price") * col("quantity"))
    .groupBy("city")
    .agg(
        max("order_value").alias("highest_orderValue"),
        min("order_value").alias("lowest_orderValue"),
    )
    .withColumn("difference", col("highest_orderValue") - col("lowest_orderValue"))
)

qsn9.show()
sql = """
SELECT 
    city,
    MAX(price * quantity) AS highest_order_value,
    MIN(price * quantity) AS lowest_order_value,
    MAX(price * quantity) - MIN(price * quantity) AS difference
FROM orders
GROUP BY city
"""